# Lab 1: Tokenization & Cost Analysis

Four ideas to take away:
1. **LLMs predict the next token** — generation is a loop: score every token, sample, append, repeat
2. **Tokens ≠ words** — LLMs see subword units, and counts vary by text type
3. **Tokens cost money** — pricing differs dramatically across models
4. **Context windows scale badly** — attention is O(n²)

Run every cell top-to-bottom and observe the outputs.

In [ ]:
!uv pip install tiktoken plotly transformers torch -q

---
## Part 0: Next-Word Prediction — The Full Pipeline

Every LLM does exactly one thing: **predict the next token**. But what does that look like step by step?

We'll make every stage visible:

1. **Text → Token IDs** — the tokenizer converts text to integers
2. **Token IDs → Logits** — the model scores every vocabulary token
3. **Logits → Probabilities** — softmax turns raw scores into a distribution
4. **Greedy selection** — pick the highest-probability token and decode it back to text
5. **Loop** — repeat to generate a full sentence

We'll use **[LFM2.5-350M](https://huggingface.co/LiquidAI/LFM2.5-350M)** — a 350M-parameter multilingual model that runs on CPU with no API key.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "LiquidAI/LFM2.5-350M"
print(f"Loading {MODEL_ID} ...  (first run downloads ~700 MB)")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model     = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16)
model.eval()
print("Model ready.\n")
print(f"Vocabulary size: {tokenizer.vocab_size:,} tokens")

### Step 1 — Text → Tokens → IDs

The tokenizer breaks text into **subword pieces** and maps each piece to an integer ID.  
The model never sees letters — only a list of integers from `[0, vocab_size)`.

In [ ]:
prompt = "The sky is"

token_ids = tokenizer.encode(prompt)
tokens    = [tokenizer.decode([i]) for i in token_ids]

print(f"Prompt : {repr(prompt)}")
print(f"\n{'Token':<20} {'ID':>8}")
print("-" * 30)
for tok, tid in zip(tokens, token_ids):
    print(f"{repr(tok):<20} {tid:>8}")
print(f"\nTotal: {len(token_ids)} token(s) → tensor shape passed to model: [batch=1, seq_len={len(token_ids)}]")

### Step 2 — Token IDs → Logits

The model's forward pass takes those IDs and outputs **one raw score per vocabulary token** for the next position.  
These scores are called **logits** — unbounded numbers. A higher logit means the model considers that token more likely.  
The spread between logits encodes confidence: a tight cluster means uncertainty, a large gap means the model is sure.

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt")
with torch.no_grad():
    logits = model(**inputs).logits  # shape: [batch, seq_len, vocab_size]

last_logits = logits[0, -1]  # scores for the NEXT token position only

print(f"Logit tensor shape : {list(logits.shape)}")
print(f"  → [batch=1, seq_len={logits.shape[1]}, vocab_size={logits.shape[2]:,}]")
print(f"\nThe model scored every one of the {logits.shape[2]:,} vocabulary tokens.")
print(f"\nTop raw logit : {last_logits.max().item():.2f}")
print(f"Min raw logit : {last_logits.min().item():.2f}")

### Step 3 — Logits → Probabilities (Softmax)

Softmax converts the raw logit scores into a proper probability distribution:  
all values in `[0, 1]`, summing to exactly **1.0**.

This is the **"bag of balls"** from the slides — one probability for every token in the vocabulary.

In [ ]:
import matplotlib.pyplot as plt

probs = torch.softmax(last_logits, dim=-1)

print(f"Sum of all {probs.shape[0]:,} probabilities: {probs.sum().item():.6f}")  # should be ≈ 1.0

top_k      = 10
top        = torch.topk(probs, top_k)
top_tokens = [tokenizer.decode([i]) for i in top.indices]
top_probs  = [p.item() * 100 for p in top.values]

print(f"\nTop {top_k} candidates for the next token after {repr(prompt)}:")
print(f"{'Token':<20} {'Probability':>12}")
print("-" * 34)
for tok, p in zip(top_tokens, top_probs):
    print(f"{repr(tok):<20} {p:>11.2f}%")

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar([repr(t) for t in top_tokens], top_probs, color="#00C9A7", edgecolor="#1C355E")
ax.bar_label(bars, fmt="%.1f%%", padding=3)
ax.set_title(f"Next-token probability distribution  |  prompt: {repr(prompt)}")
ax.set_ylabel("Probability (%)")
ax.set_xlabel("Token")
plt.tight_layout()
plt.show()

### Step 4 — Greedy Decoding: Pick the Next Token

The simplest strategy: `argmax` — always take the token with the highest probability.  
That is **greedy decoding**. We pick it, decode the ID back to text, and have our first generated token.

In [ ]:
next_token_id = probs.argmax().item()
next_token    = tokenizer.decode([next_token_id])

print(f"Greedy selection:")
print(f"  Token ID  : {next_token_id}")
print(f"  Token     : {repr(next_token)}")
print(f"  Prob      : {probs[next_token_id].item() * 100:.2f}%")
print(f"\n  {repr(prompt)} + {repr(next_token)} → {repr(prompt + next_token)}")
print(f"\nThat's one step. Feed the new sequence back in to get the next token — and so on.")

### Step 5 — The Autoregressive Loop

Now watch the loop run automatically. Each iteration repeats Steps 1–4 until the model predicts the stop token `<EOS>`.

In [ ]:
def generate_step_by_step(prompt, max_new_tokens=8):
    print(f"Starting: '{prompt}'\n")
    current = prompt
    for step in range(1, max_new_tokens + 1):
        inputs  = tokenizer(current, return_tensors="pt")
        with torch.no_grad():
            logits = model(**inputs).logits[0, -1]
        step_probs = torch.softmax(logits, dim=-1)
        top        = torch.topk(step_probs, 3)
        chosen     = tokenizer.decode([top.indices[0]])
        top_str    = ", ".join(
            f"'{tokenizer.decode([i])}' {p*100:.1f}%"
            for p, i in zip(top.values, top.indices)
        )
        print(f"Step {step}: top candidates -> [{top_str}]  picked '{chosen}'")
        current += chosen
        if chosen.strip() in [".", "!", "?"]:
            break
    print(f"\nFinal output: '{current}'")

generate_step_by_step(prompt)

In [ ]:
def top_next_tokens(prompt, top_n=10):
    """Convenience wrapper: runs Steps 1–3 for any prompt and prints the top-n candidates."""
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        logits = model(**inputs).logits[0, -1]
    p = torch.softmax(logits, dim=-1)
    top = torch.topk(p, top_n)
    print(f"\nPrompt: {repr(prompt)}")
    print(f"{'Token':<20} {'Probability':>12}")
    print("-" * 34)
    for prob, idx in zip(top.values, top.indices):
        word = tokenizer.decode([idx])
        print(f"{repr(word):<20} {prob.item()*100:>11.2f}%")

# Try changing these prompts — notice how the distribution shifts
top_next_tokens("The sky is")
top_next_tokens("The capital of Saudi Arabia is")

In [ ]:
# --- YOUR TURN ---
# Change the prompt below and re-run the cell. Watch the probabilities shift.
YOUR_PROMPT = "Artificial intelligence will"

top_next_tokens(YOUR_PROMPT)
generate_step_by_step(YOUR_PROMPT)

---
## Part 1: Tokens ≠ Words

The tokenizer breaks text into **subword units** — not characters, not whole words.
Watch how the same encoder splits English prose, code, and Arabic differently.

In [ ]:
import tiktoken

encoder = tiktoken.encoding_for_model("gpt-4")

samples = {
    "English prose": "The transformer architecture revolutionized natural language processing.",
    "Python code":   "def calculate_sum(a, b):\n    return a + b",
    "Multilingual":  "Bonjour! 你好! Today we're discussing tokens.",
    "Arabic":        "اسم المستخدم أحمد وعمره 30 سنة ويسكن في الرياض.",
}

print(f"{'Sample':<20} {'Chars':>6} {'Tokens':>7} {'Ratio':>7}   Breakdown")
print("-" * 80)
for name, text in samples.items():
    tokens = encoder.encode(text)
    pieces = [encoder.decode([t]) for t in tokens]
    ratio  = len(tokens) / len(text)
    print(f"{name:<20} {len(text):>6} {len(tokens):>7} {ratio:>7.2f}   {pieces}")

**Key observation:** Arabic and multilingual text produce far more tokens per character than plain English — because BPE was trained mostly on English text.

---
## Part 2: Tokens Cost Money

Every token in and out costs real money. Let's see what processing a 10-page document costs across different models.

In [ ]:
# Pricing reference (USD per 1M tokens, approximate early 2025)
PRICING = {
    "GPT-4-Turbo":      {"input": 10.00, "output": 30.00},
    "GPT-4o":           {"input":  2.50, "output": 10.00},
    "GPT-3.5-Turbo":    {"input":  0.50, "output":  1.50},
    "Claude-3.5-Sonnet":{"input":  3.00, "output": 15.00},
    "Gemini-1.5-Pro":   {"input":  1.25, "output":  5.00},
}

def estimate_cost(text, model_name, expected_output_tokens=500):
    enc = tiktoken.encoding_for_model("gpt-4")
    input_tokens = len(enc.encode(text))
    prices = PRICING[model_name]
    input_cost  = (input_tokens          / 1_000_000) * prices["input"]
    output_cost = (expected_output_tokens / 1_000_000) * prices["output"]
    return {
        "model":        model_name,
        "input_tokens": input_tokens,
        "input_cost":   input_cost,
        "output_cost":  output_cost,
        "total_cost":   input_cost + output_cost,
    }

# Simulate a 10-page document (~5,000 words)
doc = ("The rapid advancement of artificial intelligence has transformed "
       "industries across the globe. Organizations are increasingly adopting "
       "machine learning systems for automation, analysis, and decision-making. ") * 170

print(f"Document: {len(doc):,} chars, ~{len(doc.split()):,} words\n")
print(f"{'Model':<22} {'Input tokens':>13} {'Input $':>9} {'Output $':>9} {'Total $':>9}")
print("-" * 66)
for model in PRICING:
    r = estimate_cost(doc, model, expected_output_tokens=500)
    print(f"{r['model']:<22} {r['input_tokens']:>13,} {r['input_cost']:>9.4f} {r['output_cost']:>9.4f} {r['total_cost']:>9.4f}")

**Key observation:** GPT-4-Turbo costs ~20× more than GPT-3.5-Turbo for the same document. Model choice matters enormously at scale.